In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
print("google.colab" in sys.modules)

True


In [ ]:
import os

os.listdir('/content/drive/MyDrive')

['Colab Notebooks', 'neural_style_transfer.zip', 'nst_checkpoints']

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer,TrainingArguments , T5ForConditionalGeneration



In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [ ]:
train_data.sample(15)

,id,dialogue,summary
7516,13820574,Amber: <file_photo>\r\nSophia: O wow!!\r\nIsla...,Amber looked stunning during her prom. Sophia ...
1324,13682120-1,Emily: Finally on the flybe plane that has bee...,Emily is on a plane that is delayed for an hour.
6448,13829926,Mike: A cigarette after math?\r\nIan: Sure!\r\...,Mike and Ian will have a cigarette after math.
5343,13682246,Bridget: Some workout today?\r\nKristen: Can’t...,"Kristen has hangover today, so she can't join ..."
5319,13820080,Turner: which gym do you use guys\r\nLangston:...,Turner wants to join a gym. Mortimer recommend...
6808,13728571,Ollie: Where is the store?\r\nJake: where its ...,Ollie can't find the store. He needs Jake's he...
3290,13828750,Camilla: Have you asked him finally?\r\nAva: y...,Ava regrets asking her partner about his relat...
6427,13728074,Hilary: Can you pick up Sean?\r\nBill: let me ...,Bill will pick up Sean on Hilary's request.
3301,13681348,Carly: I'm home now and in all evening\r\nCarl...,Don will pick the car up in the morning.
6695,13731026,Bob: What time is it in London?\r\nDrella: 1 a...,It is 1 am in London.


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
# Data pre-processing

import re #regular expressions

def clean_data(text):
  text = re.sub(r"\r\n"," ",text) #lines
  text = re.sub(r"\s+"," ",text) #spaces
  text = re.sub(r"<.*?>"," ",text) # html tags
  text = text.strip().lower()
  return text


In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)


In [ ]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [ ]:
# Tokenization
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:

def tokenize(data):

    inputs = tokenizer(
        data["dialogue"],
        padding="max_length",
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        data["summary"],
        padding="max_length",
        max_length=150,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]

    return inputs


In [ ]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()



In [ ]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
# input ids - dialogue => token_ids
# 1 => eos ending of sequence

#attention masks

#labels-target => summary token

In [ ]:
len(train_dataset[0]["input_ids"])

512

In [ ]:
len(train_dataset[0]["labels"])

150

In [ ]:
type(train_dataset)

list

In [ ]:
type(val_dataset)

list

In [ ]:
# working with model

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# fine-tune
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model.to(device)


device: cuda
GPU: Tesla T4


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
#training arguments

training_args = TrainingArguments(
    output_dir= "./results",
    num_train_epochs=3,
    weight_decay=0.01,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=400


)

In [ ]:
trainer = Trainer(
    model = model,
    args =training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.403373,0.360595
2,0.372630,0.352494
3,0.363209,0.351039


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4002, training_loss=0.7553805496054015, metrics={'train_runtime': 698.2412, 'train_samples_per_second': 17.186, 'train_steps_per_second': 5.732, 'total_flos': 1624101617664000.0, 'train_loss': 0.7553805496054015, 'epoch': 3.0})

In [ ]:
!pip install huggingface_hub

from huggingface_hub import login
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# 2. Path where Colab saved your trained model
SAVE_PATH = "./saved_summary_model"

# 3. Save locally inside Colab's temporary session
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# 4. Upload directly to Hugging Face Hub
model.push_to_hub("AlishaShaikh20/text-summarizer-t5")
tokenizer.push_to_hub("AlishaShaikh20/text-summarizer-t5")

print("Upload complete! Check: https://huggingface.co/AlishaShaikh20/text-summarizer-t5")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mdjhei3/model.safetensors:   0%|          |  552kB /  242MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Upload complete! Check: https://huggingface.co/AlishaShaikh20/text-summarizer-t5


In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)

    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )

    # Move inputs to GPU/CPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.to(device)
    model.eval()

    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(
        targets[0],
        skip_special_tokens=True
    )

    return summary

In [ ]:
test_dialogue = """Customer: Hi, I ordered a laptop last week but it still hasn't arrived.

Agent: I'm sorry to hear that. Can you provide your order number?

Customer: Yes, it's ORD-45892.

Agent: Thank you. Let me check the status.

Agent: I can see the package was delayed due to weather conditions and is expected to arrive in two days.

Customer: Okay, thanks for the update. Can I get a notification when it ships?

Agent: Yes, I've enabled SMS and email notifications for your order.

Customer: Great, thank you.

Agent: You're welcome. Is there anything else I can help you with today?

Customer: No, that's all."""

In [ ]:
summary = summarize_dialogue(test_dialogue)
print("summary: ",summary)

summary:  customer ordered a laptop last week but it still hasn't arrived. agent has enabled sms and email notifications for the order.
